In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
historical_data = pd.read_csv('../data/grievance_dataset_history.csv')
new_grievance = pd.read_csv('../data/02_input_anomaly_detection.csv')

In [3]:
historical_data['recvd_date'] = pd.to_datetime(historical_data['recvd_date'], format='mixed', errors='coerce')
new_grievance['recvd_date'] = pd.to_datetime(new_grievance['recvd_date'], format='mixed', errors='coerce')

In [4]:
# !pip install transformers torch pandas

In [ ]:
import pandas as pd
from transformers import pipeline

TEXT_COL = "grievance_text_processed"   
DEVICE = -1 

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=DEVICE
)

candidate_labels = [
    "valid public service complaint or grievance",
    "spam, advertisement, or scam message",
    "abusive, offensive, or threatening language",
    "irrelevant or meaningless text"
]

C:\Users\kinge\anaconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Loading weights:   0%|                                                                                                                 | 0/515 [00:00<?, ?it/s]


Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 515/515 [00:00<00:00, 7307.69it/s]

In [ ]:
def get_decision(text):
    text = "" if pd.isna(text) else str(text).strip()

    # very short text: usually reject
    if len(text.split()) <= 2:
        return "REJECT"

    result = classifier(text, candidate_labels, multi_label=True)
    scores = dict(zip(result["labels"], result["scores"]))

    valid = scores.get("valid public service complaint or grievance", 0)
    spam = scores.get("spam, advertisement, or scam message", 0)
    abuse = scores.get("abusive, offensive, or threatening language", 0)
    irrelevant = scores.get("irrelevant or meaningless text", 0)

    # Reject if any bad category is strong enough and stronger than valid
    if spam >= 0.80 and spam > valid:
        return "REJECT"
    if irrelevant >= 0.75 and irrelevant > valid:
        return "REJECT"
    if abuse >= 0.85 and abuse > valid:
        return "REJECT"

    return "ACCEPT"

new_grievance["decision"] = new_grievance[TEXT_COL].apply(get_decision)

                            grievance_text_processed decision
0  railways railway board miscellaneous railway b...   ACCEPT


In [7]:
# new_grievance["decision"] = "ACCEPT"

In [8]:
def apply_blocking(new_record, db_df, days_window=30):
    lower_bound = new_record['recvd_date'] - pd.Timedelta(days=days_window)
    upper_bound = new_record['recvd_date']

    time_condition = (
        (db_df['recvd_date'] >= lower_bound) &
        (db_df['recvd_date'] <= upper_bound)
    )

    entity_condition = (
        (db_df['state'] == new_record['state']) &
        (db_df['_id'] == new_record['_id'])
    )

    filtered_db = db_df[time_condition & entity_condition & (new_record['decision'] == "ACCEPT")].copy()
    filtered_db.reset_index(drop=True, inplace=True)
    return filtered_db

new_record = new_grievance.iloc[0]
search_space_df = apply_blocking(new_record, historical_data)

In [9]:
if not search_space_df.empty:

    vectorizer = TfidfVectorizer(stop_words='english')

    historical_texts = search_space_df['grievance_text_processed'].tolist()

    # ensure single string
    new_record = new_grievance.iloc[0]
    new_text = [new_record['grievance_text_processed']]

    all_texts = historical_texts + new_text

    tfidf_matrix = vectorizer.fit_transform(all_texts)

    historical_vectors = tfidf_matrix[:-1]
    new_vector = tfidf_matrix[-1]

    similarities = cosine_similarity(new_vector, historical_vectors).flatten()

    max_score = similarities.max()

    print("\n--- RESULT ---")

    if max_score > 0.95:
        print(f"Duplicate Found (Score: {max_score:.4f})")
    else:
        print(f"Unique Record (Max Score: {max_score:.4f}) → Saving to CSV")

        output_path = '../data/03_input_domain_classification.csv'

        new_record_df = pd.DataFrame([new_record])
        new_record_df.to_csv(output_path, index=False)

else:
    output_path = '../data/03_input_domain_classification.csv'

    new_record_df = pd.DataFrame([new_record])
    new_record_df.to_csv(output_path, index=False)